# 17. 台灣：資料、模型與下一步

最後一章，把第二部的所有工具帶回家。台灣是地球上做統計地震
學最幸運的地方之一：板塊聚合速率每年八公分、地震多到不缺
樣本、觀測網密度世界頂級、目錄免費公開（你在第 2 章就申請
過了）。這一章用台灣的資料與台灣的研究，做四件事：盤點目錄
的家底、整理統計地震學的積累、對 0403 花蓮做一次誠實的事後
檢視，最後回答：台灣的地震預報，現在走到哪裡、下一步是什麼。

## 17.1 目錄的百年整備

第 10 章說過：目錄是一段行政史。台灣這一段特別精彩——從
1897 年台北測候所啟動觀測算起，超過一百二十年。畫出 1973 年
以來儀器目錄的全貌：

In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from gdms_toolkit import load_taiwan_catalog
from gdms_toolkit.viz import ACCENT, PALETTE, QUAKE_COLOR, apply_layout

cat = load_taiwan_catalog()
yearly = cat.set_index("time").resample("YE").size()

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.6, 0.4],
                    vertical_spacing=0.04)
big = cat[cat.ML >= 5.0]
fig.add_trace(go.Scattergl(x=big.time, y=big.ML, mode="markers",
                           name="ML ≥ 5", marker=dict(size=4, color=ACCENT,
                                                      opacity=0.5)),
              row=1, col=1)
fig.add_trace(go.Bar(x=yearly.index, y=yearly.values, name="年事件數（全目錄）",
                     marker_color=PALETTE[2]), row=2, col=1)
for date, label in [("1999-09-21", "集集"), ("2024-04-03", "0403 花蓮")]:
    for r in (1, 2):
        fig.add_vline(x=date, line_dash="dot", line_color=QUAKE_COLOR,
                      row=r, col=1)
fig.add_annotation(x="1999-09-21", y=7.4, text="集集", showarrow=False,
                   font=dict(color=QUAKE_COLOR), row=1, col=1)
fig.add_annotation(x="2024-04-03", y=7.4, text="0403", showarrow=False,
                   font=dict(color=QUAKE_COLOR), row=1, col=1)
fig.update_yaxes(title_text="規模 ML", row=1, col=1)
fig.update_yaxes(title_text="年事件數", row=2, col=1)
apply_layout(fig, title=f"台灣長期目錄總覽（1973–2025，共 {len(cat):,} 筆）",
             height=560, hovermode="x")
fig

上圖是 M≥5 的大事件——集集與 0403 花蓮的餘震序列像兩根
直柱。下圖的年事件數則幾乎與地震無關：1994 年（觸發式改
連續記錄）與 2012 年（觀測網升級）的兩次階梯，寫的是儀器史
不是地震史。第 10 章的功課在自己的資料上一目瞭然。

幾件家底值得清點。規模尺度換過三次（震度反推的 $M_H$ →
延時規模 $M_D$ → 1991 年起模擬 Wood–Anderson 的 $M_L$），
學界已把 1900–2014 的目錄均一化成 $M_w$（Chang et al. 2016,
補回 188 個早年事件、修復 1985–1991 的資料空隙）；寬頻矩
張量目錄 AutoBATS 提供 1996 年以來八千多個自動機制解，並
給出台灣的規模轉換式 $M_w = 0.87\,M_L + 0.23$——$M_L$ 在
大規模端飽和偏高，做任何跨目錄比較前先換算。中央氣象署的
地震目錄至今累積超過 67 萬筆，是這一切的基石。

## 17.2 統計地震學的積累

第 10、11 章的每一條定律，台灣都有自己的數字。整理成一張
「在地基準值」表（出處與但書見各章）：

| 量 | 台灣值 | 但書 |
|---|---|---|
| b 值（全台平均） | ≈ 1.0，陸上高、外海低 | 隨年代與規模尺度變 |
| b 值（1973–87 vs 1994 後） | 0.83 vs 0.99 | 差異來自尺度換代，非構造 |
| Omori p（集集） | 1.05 | Lee et al. 2013 |
| Omori p（池上 2022） | 0.92–1.39 | 強烈依賴擬合時間窗 |
| p 隨主震規模 | $p \approx 0.11 M_m + 0.38$ | Tsai et al. 2012 |
| Båth ΔM（706 序列） | 1.20 ± 0.73 | 餘震數 >50 時降至 0.74 |
| ETAS（全台，兩份獨立估計） | $p \approx 1.04$–$1.06$、$\alpha \approx 1.0$–$1.2$ | 1994 年後目錄、$M_c = 3.0$ |
| Mc 空間場 | 陸上 1.5–2.0、外海 2.5–3.2 | Chan & Wu 2013 |

這張表有兩個讀法。表面上它是速查表；深一層，它是第二部
方法論的總復習——每一列的「但書」欄，都是前面某一章的
核心教訓：尺度換代（10 章）、時間窗敏感（10 章）、平均值
不等於個案預測力（10 章）、參數要配目錄斷代（11 章）。
另外兩個值得記住的台灣發現：2022 池上序列的**前震 b 值
（約 0.5–0.6）顯著低於餘震（約 0.85）**，與國際上「前震
b 值偏低」的候選判別一致——但這是回溯分析，不是即時判定;
以及集集餘震的異常豐富（GR 外推的最大餘震幾乎與主震同級）,
提醒 Båth「常數」在個案上的脆弱。

## 17.3 0403 花蓮：一次誠實的事後檢視

第一部第 7 章對 0403 花蓮做過四類觀測資料的對照，結論是
乾淨的訊號只有波形與餘震。現在有了第二部的工具，可以問
更精確的問題：**如果當時台灣有一套完整的預報系統，它會
說什麼？**

先看長期。用 0403 **之前**的目錄（1973–2024/03）做一張
PPE 式平滑地震度地圖，疊上 0403 震央：

In [ ]:
pre = cat[(cat.time < "2024-04-01") & (cat.ML >= 5.0)]
step = 0.1
lons = np.arange(119.0, 123.5, step)
lats = np.arange(21.0, 26.0, step)
LON, LAT = np.meshgrid(lons, lats)
dens = np.zeros_like(LON)
for lo, la, m in pre[["longitude", "latitude", "ML"]].to_numpy():
    r2 = ((LON - lo) * 111 * np.cos(np.radians(la))) ** 2 \
         + ((LAT - la) * 111) ** 2
    dens += (m - 4.9) * (1 / (np.pi * (15 ** 2 + r2)) + 1e-4)

fig = go.Figure(go.Heatmap(x=lons, y=lats, z=np.log10(dens),
                           colorscale="Blues",
                           colorbar=dict(title="log₁₀ 相對率")))
fig.add_trace(go.Scatter(x=[121.56], y=[23.77], mode="markers",
                         name="0403 花蓮 M7.2",
                         marker=dict(symbol="x", size=14, color=QUAKE_COLOR,
                                     line=dict(width=3))))
apply_layout(fig, title="只用 0403 之前的目錄：過去地震的鄰近性「看見」花蓮了嗎？",
             xaxis_title="經度", yaxis_title="緯度",
             yaxis_scaleanchor="x", hovermode="closest", height=560)
fig

答案是：看見了，但這算不上預報。花蓮外海本來就是全台地震
度最高的區域之一——長期模型本來就把最高的率放在那裡。
這正是第 12 章講過的：PPE 這類基準模型回答「哪裡」很在行,
對「何時」完全沉默。0403 落在高背景區，是基準模型的成功,
也是它的天花板。

短期呢？0403 主震前，即時目錄上沒有顯著的前震序列（這與
2022 池上不同——池上有 $M_L$ 6.6 的關山前震在前一天）。
一套 ETAS 型的 OAF 系統在 0403 **之前**不會發出任何警示，
但在主震**之後**能立刻接手：第 11 章你已經看過它的餘震率
衰減多麼服從 Omori 律——餘震機率預報、震度超越機率，技術
上完全做得出來，2025 年大埔序列的快報就是實證。中期呢？
誠實的回答是：台灣目前**沒有**作業化的中期模型可以事後
檢視——這正是下一節要談的缺口。

這個檢視最大的價值，是它把第 8 章的警告變得具體：事後回頭
看，你總能在圖上找到「跡象」；但預報系統必須在事前、按
事先定好的規則發言。0403 教我們的不是「差一點就能預測」，
而是**各時間尺度的工具各自能做什麼、不能做什麼**。

## 17.4 台灣的預報之路：現況與缺口

把台灣現有的能力攤在時間軸上：

In [ ]:
tools = [
    ("地震預警（EEW）", np.log10(3 / 86400 / 365.25), np.log10(60 / 86400 / 365.25),
     "#1baf7a", "作業中（世界前段班）"),
    ("短期預報（ETAS/OAF）", np.log10(1 / 365.25), np.log10(90 / 365.25),
     "#2a78d6", "技術就緒（大埔快報實證）"),
    ("中期預報（EEPAS 類）", np.log10(0.25), np.log10(20),
     "#e34948", "缺口（在地化進行中）"),
    ("長期危害（TEM PSHA）", np.log10(10), np.log10(500),
     "#4a3aa7", "已有兩代國家級模型"),
]
fig = go.Figure()
for i, (name, lo, hi, color, status) in enumerate(tools):
    fig.add_trace(go.Bar(y=[name], x=[hi - lo], base=[lo], orientation="h",
                         marker_color=color, opacity=0.8, name=status,
                         text=status, textposition="inside"))
fig.update_xaxes(title_text="時間尺度（年，log₁₀）",
                 tickvals=[-6, -4, -2, 0, 2],
                 ticktext=["秒–分", "小時", "天", "年", "百年"])
apply_layout(fig, title="台灣地震風險資訊工具的時間尺度地圖",
             showlegend=False, hovermode="closest", height=380)
fig

**已經很強的：預警。**1999 集集地震，速報系統在大停電中
102 秒算出位置與規模；2018 花蓮地震，17 秒發布警報、20 秒
送達民眾手機。這是台灣真正領先世界的部分——但記住第 9 章
的區辨：預警是「地震已發生、波還在路上」，跟預報是兩件事。

**正在成形的：短期統計預報。**本土化的時空 ETAS 參數有兩份
獨立估計互相印證；2025 大埔序列的快報示範了完整的作業化
流程（預訓練一小時、即時每輪 7 分鐘、輸出場址震度機率），
並罕見地把預報機率與實際觀測並列發表。

**明顯的缺口，有四個。**中期（月到十年）模型在台灣還是
空白——短期有 ETAS、長期有 PSHA，中間這段沒有在地化的
作業模型；EEPAS 的開源化與台灣在地化正由台灣團隊進行中。
預報**檢驗文化**尚未建立——台灣多為個案回溯分析，缺少
CSEP 式、事先註冊的前瞻檢驗，而第 15 章說得很清楚：台灣
測試區小、目標地震少，正是低統計功效的情境，檢驗設計要
格外用心。**即時目錄與重定位目錄的落差**——作業預報用
即時目錄，其完整度與定位品質都遜於研究用目錄，大埔快報的
作者就誠實指出某段「平靜」無法據即時目錄定論。最後是
**前兆**——中央氣象署自己的業務回顧承認，短期前兆可視為
成功的比例在兩成以下；這不是放棄的理由，而是把資源優先
投給統計預報的理由。

制度面則有第 9 章留下的兩個問題等著台灣回答：機率預報的
**權威性**由誰授予？要走義大利路線（權威但不公開、服務
民防）、紐西蘭路線（公開、與使用者共同設計）、還是美國
路線（大震後自動發布）？這些不是技術問題，但答案會決定
技術被怎麼使用。

## 17.5 第二部的終點，也是起點

這本教材從地下水位的固體潮講起，繞了一大圈，結束在地震
預報的制度設計。回頭看，兩部其實在講同一件事的兩面。
第一部教你對「看起來像前兆的異常」保持懷疑——因為訊號
太小、樣本太少、事後選擇太會騙人。第二部教你這個領域
如何把懷疑**制度化**：基準模型、前瞻檢驗、資訊增益、
統計功效——一整套讓錯誤的宣稱無所遁形的機制，以及在
這套機制下仍然存活下來的、貨真價實的可預報性：叢集。

地震預報今天能誠實說出口的成果是：我們無法告訴你下一個
大地震的時間地點，但我們能告訴你機率——而且這個機率經得
起檢驗、能接上危害與工程、正在多個國家每天運轉。台灣有
世界級的資料、世界級的預警、正在成形的短期預報，以及
一段還沒有人填上的中期空白。

那段空白，也許就留給正在讀這一頁的你。工具都在前面十六章
裡了；目錄就在 `data/cache/` 裡。動手之前，只需要記得
全書說了兩遍的那句話——一個數字算得出來，不等於它站得
住腳；讓它站住腳的方法，你現在已經會了。